### Load the mistral-7b-v0.3 model

Adapted from https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_v0.3_(7B)-Alpaca.ipynb

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    # model_name = "unsloth/mistral-7b-bnb-4bit",
    # model_name = "unsloth/tinyllama", # https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/TinyLlama_(1.1B)-Alpaca.ipynb
    # model_name = "microsoft/phi-2", # https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/TinyLlama_(1.1B)-Alpaca.ipynb
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

### Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Data preparation

In [ ]:
from datasets import load_dataset
train_dataset, test_dataset = load_dataset("json", data_files={"train": "dataset-for-finetuning-sentences-train.json", "test": "dataset-for-finetuning-sentences-test.json"}, split=["train", "test"])
print("training: " + str(train_dataset.num_rows))
print("test: " + str(test_dataset.num_rows))

In [ ]:
# the whole dataset consists of 2000 samples, 400 different scripts with 5 variations each.
# i.e. the test dataset consists of 400 samples, 80 different scripts with 5 variations each, so we pick every 5th
test_dataset = test_dataset.select(range(0, 400, 5))

In [ ]:
print(test_dataset[1]['rephrased'])

In [ ]:
print(test_dataset)

In [ ]:
prompt = '''
Below is the description of a scientific microscopy imaging experiment. Transfer this description into valid json.

### Instruction:
{}

### Input:
{}

### Response:
{}'''

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def apply_chat_template(example):
    sentence  = example['json']
    rephrased = example['rephrased']
    context   = example['context']
    example['text'] = prompt.format(context, rephrased, "// Sample script written in json:\n" + sentence) + EOS_TOKEN
    return example

column_names = list(train_dataset.features)

processed_train_dataset = train_dataset.map(
    apply_chat_template,
    remove_columns=column_names,
    desc="Applying chat template to train",
)

processed_test_dataset = test_dataset.map(
    apply_chat_template,
    remove_columns=column_names,
    desc="Applying chat template to test",
)

In [ ]:
print(processed_train_dataset[35]['text'])

### Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Set num_train_epochs = 1 for full training runs
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
        # https://www.reddit.com/r/unsloth/comments/1bnm3yd/validation_dataset/
        fp16_full_eval = True,
        per_device_eval_batch_size = 2,
        eval_accumulation_steps = 4,
        eval_strategy = "steps",
        eval_steps = 5,
        do_eval = True,
    ),
    eval_dataset = processed_test_dataset,
)

#### Show current memory stats

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# https://unsloth.ai/blog/gradient
from unsloth import unsloth_train
# trainer_stats = trainer.train() # << Buggy gradient accumulation
trainer_stats = unsloth_train(trainer)

In [ ]:
import json

with open("outputs/checkpoint-100/trainer_state.json", "r") as f:
    logs = json.load(f)

In [ ]:
train_loss = []
eval_loss = []
for l in logs['log_history']:
    step = l['step']
    try:
        train_loss.append((step, l['loss']))
    except:
        pass
    try:
        eval_loss.append((step, l['eval_loss']))
    except:
        pass

In [ ]:
import matplotlib.pyplot as plt

# Plot Training vs Validation Loss
# https://stackoverflow.com/questions/21519203/plotting-a-list-of-x-y-coordinates
plt.figure(figsize=(8, 5))
plt.plot(*zip(*train_loss), marker="", label="Training Loss", color="orange")
plt.plot(*zip(*eval_loss), marker="", label="Validation Loss", color="green")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss Over Time")
plt.legend()
plt.grid(True)
plt.show()

### Save the model in SafeTensor format

#### Adapters

In [ ]:
model.save_pretrained("results/sentence-adapters-json")  # Local saving
tokenizer.save_pretrained("results/sentence-adapters-json")

In [ ]:
import shutil
shutil.make_archive("results/Lora_Adapters_Json", 'zip', "results/sentence-adapters-json")

### Use the trained model for prediction

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "outputs/checkpoint-60",
    model_name = "results/sentence-adapters-json",
    # model_name = "results/adapters-4bnb",
    # model_name = "results/merged",
    max_seq_length = 2048, # 4096,
    dtype = None,
    load_in_4bit = True
)
FastLanguageModel.for_inference(model);



In [ ]:
from datasets import load_dataset
train_dataset, test_dataset = load_dataset("json", data_files={"train": "dataset-for-finetuning-sentences-train.json", "test": "dataset-for-finetuning-sentences-test.json"}, split=["train", "test"])

In [ ]:
prompt = '''
Below is the description of a scientific microscopy imaging experiment. Transfer this description into valid json.

### Instruction:
{}

### Input:
{}

### Response:
'''

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

In [ ]:
sample = prompt.format(test_dataset[0]['context'], test_dataset[0]['rephrased'])
print(sample)

inputs = tokenizer([sample], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache = True)
r = tokenizer.batch_decode(outputs)
result = r[0].replace("\\n", "\n")
print(result)

In [ ]:
print(model.generation_config.to_dict())

In [ ]:
sample_input = ""  # <-- Put your input here
sample = prompt.format("", sample_input)
# print(sample)

inputs = tokenizer([sample], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache = True)
r = tokenizer.batch_decode(outputs)

result = r[0].replace("\\n", "\n")
print(result)